In [ ]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/asl-alphabet/asl_alphabet_test/asl_alphabet_test/A_test.jpg
/kaggle/input/asl-alphabet/asl_alphabet_test/asl_alphabet_test/E_test.jpg
/kaggle/input/asl-alphabet/asl_alphabet_test/asl_alphabet_test/L_test.jpg
/kaggle/input/asl-alphabet/asl_alphabet_test/asl_alphabet_test/N_test.jpg
/kaggle/input/asl-alphabet/asl_alphabet_test/asl_alphabet_test/S_test.jpg
/kaggle/input/asl-alphabet/asl_alphabet_test/asl_alphabet_test/D_test.jpg
/kaggle/input/asl-alphabet/asl_alphabet_test/asl_alphabet_test/G_test.jpg
/kaggle/input/asl-alphabet/asl_alphabet_test/asl_alphabet_test/I_test.jpg
/kaggle/input/asl-alphabet/asl_alphabet_test/asl_alphabet_test/W_test.jpg
/kaggle/input/asl-alphabet/asl_alphabet_test/asl_alphabet_test/M_test.jpg
/kaggle/input/asl-alphabet/asl_alphabet_test/asl_alphabet_test/nothing_test.jpg
/kaggle/input/asl-alphabet/asl_alphabet_test/asl_alphabet_test/X_test.jpg
/kaggle/input/asl-alphabet/asl_alphabet_test/asl_alphabet_test/H_test.jpg
/kaggle/input/asl-alphabet/asl_a

In [2]:
import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import os


2025-12-10 08:04:40.858816: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1765353881.193923      47 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1765353881.401309      47 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

In [3]:
train_dir = r"/kaggle/input/asl-alphabet/asl_alphabet_train/asl_alphabet_train"   


img_height = 224
img_width = 224
batch_size = 32

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=10,
    width_shift_range=0.1,
    height_shift_range=0.1,
    brightness_range=[0.8, 1.4],
    zoom_range=0.15,
    horizontal_flip=False,
    vertical_flip=False,
    validation_split=0.2,
    shear_range=0.2

)

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(img_height, img_width),
    batch_size=batch_size,
    class_mode="categorical",
    subset="training"
)

val_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(img_height, img_width),
    batch_size=batch_size,
    class_mode="categorical",
    subset="validation"
)



print("Number of classes detected:", train_generator.num_classes)
print("Classes:", train_generator.class_indices)

total_train_images = train_generator.samples
total_val_images = val_generator.samples
print("Training images:", total_train_images)
print("Validation images:", total_val_images)


Found 69600 images belonging to 29 classes.
Found 17400 images belonging to 29 classes.
Number of classes detected: 29
Classes: {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4, 'F': 5, 'G': 6, 'H': 7, 'I': 8, 'J': 9, 'K': 10, 'L': 11, 'M': 12, 'N': 13, 'O': 14, 'P': 15, 'Q': 16, 'R': 17, 'S': 18, 'T': 19, 'U': 20, 'V': 21, 'W': 22, 'X': 23, 'Y': 24, 'Z': 25, 'del': 26, 'nothing': 27, 'space': 28}
Training images: 69600
Validation images: 17400


In [4]:
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.callbacks import ModelCheckpoint



base_model = ResNet50(
    weights="imagenet",
    include_top=False,
    input_shape=(img_height, img_width, 3)
)


x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(512, activation='relu')(x)
output = Dense(train_generator.num_classes, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=output)
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

early_stop = EarlyStopping(
    monitor='val_loss',      
    patience=3,              
    restore_best_weights=True 
)




checkpoint = ModelCheckpoint(
    'resnet50_asl.h5',
    monitor='val_loss',
    save_best_only=True,
    save_weights_only=False
)

history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=20,
    callbacks=[early_stop, checkpoint]
)



I0000 00:00:1765353940.298040      47 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13942 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1765353940.298697      47 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13942 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


/usr/local/lib/python3.11/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/20


I0000 00:00:1765353976.827904     124 service.cc:148] XLA service 0x785c58005640 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1765353976.829626     124 service.cc:156]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1765353976.829647     124 service.cc:156]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1765353981.393007     124 cuda_dnn.cc:529] Loaded cuDNN version 90300
I0000 00:00:1765354001.373152     124 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


...

2175/2175 ━━━━━━━━━━━━━━━━━━━━ 1776s 791ms/step - accuracy: 0.8605 - loss: 0.4606 - val_accuracy: 0.9121 - val_loss: 0.3040
Epoch 2/20
2175/2175 ━━━━━━━━━━━━━━━━━━━━ 0s 457ms/step - accuracy: 0.9810 - loss: 0.0609

2175/2175 ━━━━━━━━━━━━━━━━━━━━ 1229s 565ms/step - accuracy: 0.9810 - loss: 0.0609 - val_accuracy: 0.9220 - val_loss: 0.2510
Epoch 3/20
2175/2175 ━━━━━━━━━━━━━━━━━━━━ 0s 467ms/step - accuracy: 0.9889 - loss: 0.0369

2175/2175 ━━━━━━━━━━━━━━━━━━━━ 1250s 575ms/step - accuracy: 0.9889 - loss: 0.0369 - val_accuracy: 0.9411 - val_loss: 0.1890
Epoch 4/20
2175/2175 ━━━━━━━━━━━━━━━━━━━━ 1231s 566ms/step - accuracy: 0.9925 - loss: 0.0254 - val_accuracy: 0.2776 - val_loss: 31.9003
Epoch 5/20
2175/2175 ━━━━━━━━━━━━━━━━━━━━ 0s 459ms/step - accuracy: 0.9906 - loss: 0.0312

2175/2175 ━━━━━━━━━━━━━━━━━━━━ 1231s 566ms/step - accuracy: 0.9906 - loss: 0.0312 - val_accuracy: 0.9525 - val_loss: 0.1750
Epoch 6/20
2175/2175 ━━━━━━━━━━━━━━━━━━━━ 1242s 571ms/step - accuracy: 0.9936 - loss: 0.0225 - val_accuracy: 0.8806 - val_loss: 0.5633
Epoch 7/20
2175/2175 ━━━━━━━━━━━━━━━━━━━━ 0s 453ms/step - accuracy: 0.9952 - loss: 0.0150

2175/2175 ━━━━━━━━━━━━━━━━━━━━ 1219s 560ms/step - accuracy: 0.9952 - loss: 0.0150 - val_accuracy: 0.9675 - val_loss: 0.1390
Epoch 8/20
2175/2175 ━━━━━━━━━━━━━━━━━━━━ 1223s 562ms/step - accuracy: 0.9945 - loss: 0.0175 - val_accuracy: 0.9547 - val_loss: 0.1881
Epoch 9/20
2175/2175 ━━━━━━━━━━━━━━━━━━━━ 1256s 577ms/step - accuracy: 0.9966 - loss: 0.0120 - val_accuracy: 0.9383 - val_loss: 0.2442
Epoch 10/20
2175/2175 ━━━━━━━━━━━━━━━━━━━━ 1248s 574ms/step - accuracy: 0.9960 - loss: 0.0135 - val_accuracy: 0.9661 - val_loss: 0.1499
